In [1]:
from pathlib import Path
import pandas as pd
import ollama
import json
from typing import Literal, get_args
from pydantic import BaseModel
import re
from playwright.async_api import async_playwright
import win32com.client as win32

## Setting up

In [2]:
main_folder = Path(r"C:\Privé\Jumbo receipts") #Empty folder where receipts will be saved
sender_email = 'example@hotmail.com'
recipient = 'example@hotmail.com'
number_of_receipts = 2                         #Number of receipts the script analyses

## Download receipts of website

This script automatically downloads the most recent receipts of the Jumbo.com website. I had no experence with these libraries, therefore I had much help from Claude for this section, although I did make sure to understand each line. 

In [3]:
OUT = main_folder

pw = await async_playwright().start()
browser = await pw.chromium.connect_over_cdp("http://localhost:9222")
page = browser.contexts[0].pages[0]

# Rows look like "Dinsdag 17 november 09:09"
date_re = re.compile(r"(Maandag|Dinsdag|Woensdag|Donderdag|Vrijdag|Zaterdag|Zondag)\s+\d{1,2}\s+\w+")
await page.wait_for_timeout(1000)

In [4]:
async def list_orders() -> list:
    """List the orders (date labels) visible on the page, newest first."""
    rows = page.get_by_text(date_re)
    return [(await rows.nth(i).inner_text()).strip() for i in range(await rows.count())]

async def open_order(index: int) -> str:
    """Open the order at this position (0 = newest) from the orders list."""
    await page.get_by_text(date_re).nth(index).click()
    await page.wait_for_url(re.compile(r"/bonnetjes/"))
    await page.wait_for_load_state("networkidle")
    return "opened: " + (await page.locator("h1").first.inner_text()).strip()

async def save_screenshot() -> str:
    """Save a screenshot of the receipt card, named after the date."""
    await page.bring_to_front()
    await page.wait_for_load_state("networkidle")

    # Wait until the receipt's own "Producten" heading exists (the menu has one too)
    prod = page.get_by_text("Producten", exact=True)
    for _ in range(20):
        if await prod.count() >= 2:
            break
        await page.wait_for_timeout(500)
    else:
        raise RuntimeError("Receipt did not load (only the menu 'Producten' found)")

    title = (await page.locator("h1").first.inner_text()).strip()
    name = re.sub(r"[^\w\-]+", "_", title)
    path = OUT / f"{name}.png"

    receipt = prod.last.locator("xpath=ancestor::*[5]") #The number 5 was found by testing, it refers to the receipt part of the page
    box = await receipt.evaluate(
        """e => { const r = e.getBoundingClientRect();
                  return {x: r.x + scrollX, y: r.y + scrollY, width: r.width, height: r.height}; }"""
    )
    print("clip box:", box)
    if box["height"] < 500 or box["width"] > 800:
        raise RuntimeError(f"Wrong element captured: {box}")

    await page.screenshot(path=path, full_page=True, clip=box, animations="disabled")
    return f"saved {name}.png"

async def go_back() -> str:
    """Return to the orders list."""
    await page.go_back()
    await page.get_by_text(date_re).first.wait_for()
    return "back on list"

In [5]:
async def download_receipts(number_of_receipts):
    orders = await list_orders()
    n = min(number_of_receipts, len(orders)) #I haven't extended the script to let it load more receipts than are loaded on the original page

    for i in range(n):
        try:
            await open_order(i)
            await save_screenshot()
        except Exception as e:
            print(f"order {i} failed: {e}")
        finally:
            await go_back()

## Construct a dataframe of all items in all receipts

I got help from Claude on downloading/installing/understanding ollama, also with the syntax that it requires.

In [6]:
PROMPT = """Read this Jumbo supermarket receipt.

Extract:
1. Every product listed under the heading "Producten". For each product give its name, its quantity, and the total price in the right-hand column. If a product has a line like "2 X 1,00" below it, the quantity is 2 and the total price is the number on the right of that line. Otherwise the quantity is 1.
2. The purchase date, found at the very bottom of the receipt.

Ignore "Totaal (incl. BTW)", "Totaal korting", the "Jumbo Extra's" box, the stamps (zegels) and the BTW table.

Reply with only valid JSON, no other text, in this shape:
{"products": [{"name": "...", "quantity": 1, "total_price": "0,00"}], "date": "DD-MM-YYYY"}
"""

def get_receipt_info(file):
    """Returns a dataframe of all products bought, what quantity at which price and on which date."""
    
    response = ollama.chat(
        model='qwen2.5vl',
        format='json',
        options={'temperature': 0},
        messages=[{
            'role': 'user',
            'content': PROMPT,
            'images': [str(file)],}],)

    data = json.loads(response['message']['content'])

    return pd.DataFrame({
        'item': [p['name'] for p in data['products']],
        'quantity': [int(p['quantity']) for p in data['products']],
        'total_price': [float(str(p['total_price']).replace(',', '.')) for p in data['products']],
        'date': pd.to_datetime(data['date'], format='%d-%m-%Y'),})

def dataframe_of_purchases(folder_of_receipts):
    """Wrote this function myself"""
    dfs = []
    
    for file in folder_of_receipts.glob("*.png"):
        if file.is_file():
            dfs.append(get_receipt_info(file))

    df = pd.concat(dfs, ignore_index=True)
    
    #Used Claude to construct an efficient one liner, this line is for including discounts on products
    is_actie = df['item'].str.upper().str.startswith('ACTIE')
    df = (df.assign(total_price=df.groupby((~is_actie).cumsum())['total_price'].transform('sum').round(2))
            [~is_actie].reset_index(drop=True))

    return df

## Categorize all items bought

I use dutch here since the receipts are also in dutch, and I figured I would get better results in this way.

In [7]:
class CategoryResult(BaseModel):
    category: Literal[
        'brood',
        'eieren',
        'fruit',
        'kwark',
        'beleg op brood',
        'schoonmaak middelen',
        'noten',
        'lekkernijen',
        'toiletspullen',
        'avondeten',
        'overig',
    ]
category_list = get_args(CategoryResult.model_fields['category'].annotation) #makes a list of the categories

#Provided a small setup, then had Claude construct the prompt, then altered a few things to make it better
SYSTEM_PROMPT = """Je bent een supermarkt producten classificator voor de Jumbo. Je krijgt de naam van één product van een kassabon. De namen zijn vaak afgekort of een merk- of rasnaam. Categoriseer het product in precies één categorie.

Categorieën:
- 'brood': brood en broodachtige producten. Herkenningswoorden: tijger, tijgerbrood, bruin, volkoren, wit, heel, half, kaiser, afbakbroodjes, pistolets, wraps, crackers, beschuit.
- 'eieren': eieren. Herkenningswoorden: scharrel, vrije uitloop, biologisch, 'M/L', 'st' (stuks) in combinatie met een getal, zoals 'Scharrel M/L 12st'.
- 'fruit': alleen fruit. Ook rasnamen van appels en peren: Kanzi, Elstar, Jonagold, Pink Lady, Conference. Verder bananen, mandarijnen, druiven, aardbeien, blauwe bessen, sinaasappels, kiwi's. Groenten zijn GEEN fruit.
- 'kwark': kwark en skyr. Herkenningswoorden: Camp, Optimel kwark, Jumbo kwark, Griekse kwark.
- 'beleg op brood': wat je op brood legt of smeert. Herkenningswoorden: kaas, jong, jong belegen, belegen, oud, plakken kaas, '30+', '48+', ham, kipfilet beleg, salami, hummus, pindakaas, hagelslag, jam, choco, smeerkaas, Philadelphia.
- 'schoonmaak middelen': allesreiniger, afwasmiddel, wasmiddel, spons, vuilniszakken, toiletreiniger, wc-eend.
- 'noten': amandelen, cashew, walnoten, pinda's, notenmix.
- 'lekkernijen': snoep, koek, chips, chocolade, ijs, frisdrank, gebak.
- 'toiletspullen': shampoo, douchegel, tandpasta, deodorant, wc-papier, scheerspullen.
- 'avondeten': alles voor een warme maaltijd: alle groenten (komkommer, wortelen, paprika, sla, ui, tomaat), vlees, kip, vis, pasta, rijst, aardappelen, roerbakmixen, pesto, pastasaus, soep en kant-en-klaarmaaltijden.
- 'overig': alleen als het product in geen andere categorie past (bijvoorbeeld energy gels, batterijen, kaarsen).

Regels:
- Kies altijd precies één categorie en schrijf de naam exact zoals hierboven.
- Pesto en pastasaus zijn 'avondeten', ook al zitten ze in een potje.
- Groenten, zoals komkommer en wortelen, zijn 'avondeten' en niet 'fruit'.
- Kaas is 'beleg op brood'. Afkortingen zoals 'Jong Bel 30+' betekenen jong belegen kaas.
- Gebruik 'overig' alleen als je echt geen andere categorie kunt kiezen.
- Antwoord alleen met de categorie, geen uitleg.

Voorbeelden:
- 'Shampoo Men 2-In-1' -> toiletspullen
- 'Jumbo Halfvolle Kwark' -> kwark
- 'Elstar Appels' -> fruit
- 'Jumbo Courgette' -> avondeten
- 'Jumbo Pindakaas' -> beleg op brood
- 'Jumbo Oud 48+ Plakken' -> beleg op brood
- 'Bruin Tijger Half' -> brood
"""

def categorize(df_products):
    """"Asked Claude for help on the syntax again."""
    df = df_products.copy()
    categories = []
    
    for _, row in df.iterrows():
        try:
            r = ollama.chat(
                model="llama3.2",
                messages=[{"role": "system", "content": SYSTEM_PROMPT},{"role": "user", "content": row['item']},],
                format=CategoryResult.model_json_schema(),options={"temperature": 0},)
            
            categories.append(CategoryResult.model_validate_json(r["message"]["content"]).category)        
        except Exception as e:
            print(f"Error: {e}")
            categories.append("Error")
    df["Category"] = categories
    return df

## Make a small summary per category

In [8]:
def summary_per_category(df_categorized, category):
    """Gives a summary of a certain category of products. The summary includes per category: total amount spent, fraction of all products spent, 
    average price per item, total number of items bought, most common item bought, most expensive item bought. Wrote this function myself.
    """
    
    df = df_categorized.copy()
    df_category = df[df['Category'] == category].copy()

    total_spend = df['total_price'].sum()

    #Stats for category
    total_spend_category = df_category['total_price'].sum()
    fraction_spend_category = total_spend_category/total_spend

    total_qty = df_category['quantity'].sum()
    average_price = total_spend_category / total_qty if total_qty > 0 else 0

    modes = df_category['item'].mode()
    most_common = modes[0] if not modes.empty else None

    
    df_category['price per item'] = df_category['total_price'] / df_category['quantity']
    
    valid_prices = df_category['price per item'].dropna()
    most_expensive = (df_category.loc[valid_prices.idxmax(), 'item'] if not valid_prices.empty else None)


    return pd.DataFrame({'category': category, "total spending": total_spend_category, "fraction of spending": fraction_spend_category, 
            "average price per item": average_price, "total number of items bought": total_qty, 
            "most common item bought": most_common, "most expensive item bought": most_expensive}, index=[0])

def create_final_summary(df_categorized, category_list):
    summaries = []
    for i in range(len(category_list)):
        summaries.append(summary_per_category(df_categorized, category_list[i]))
    
    return pd.concat(summaries, ignore_index=True)

## Send email with final summary

In [9]:
def send_summary_email(final_summary, number_of_receipts, sender_email, recipient):
    """Had the help of Claude with the construction of the email, since I was not familiar with this before."""
    outlook = win32.Dispatch("Outlook.Application")
    mail = outlook.CreateItem(0)

    for account in outlook.Session.Accounts:
        if account.SmtpAddress.lower() == sender_email.lower():
            mail._oleobj_.Invoke(*(64209, 0, 8, 0, account))
            break
    else:
        raise ValueError(f"Outlook account '{sender_email}' not found.")

    mail.To = recipient
    mail.Subject = "Jumbo receipts summary"

    html_table = final_summary.round(2).to_html(index=False, border=1)
    mail.HTMLBody = f"""
    <p>Here is the summary of the last {number_of_receipts} Jumbo receipts:</p>
    {html_table}
    """

    mail.Display()
    #mail.Send()

## Full run

In [10]:
async def run_all(main_folder, number_of_receipts, category_list):

    await download_receipts(number_of_receipts)                                    #Downloads receipts and puts them in the main_folder
    df_receipts = dataframe_of_purchases(main_folder)                              #Reads all receipts in the main_folder and puts them into a dataframe
    receipts_categorized = categorize(df_receipts)                                 #Categorizes all items in the dataframe of items
    final_summary = create_final_summary(receipts_categorized, category_list)      #Creates the final summary dataframe
    send_summary_email(final_summary, number_of_receipts, sender_email, recipient) #Sends the email with the final summary

In [11]:
await run_all(main_folder, number_of_receipts, category_list)

clip box: {'x': 32, 'y': 359.9250183105469, 'width': 585, 'height': 1611.7750244140625}
clip box: {'x': 32, 'y': 359.9250183105469, 'width': 585, 'height': 1415.7750244140625}
